# 🤖 Tech Challenge Fase 3 — Fase 2: Fine-tuning com QLoRA

> **⚡ Otimizado para Colab Free (T4 16 GB):** QLoRA 4-bit + TinyLlama 1.1B

**Pipeline:**
```
1. Instalação de dependências (trl, peft, bitsandbytes, transformers)
2. Verificação da GPU
3. Carregamento do dataset Alpaca JSONL (saída do notebook 01)
4. Formatação do prompt no template Alpaca
5. Carregamento do modelo TinyLlama com quantização 4-bit
6. Configuração do LoRA (PEFT)
7. Treinamento supervisionado com SFTTrainer
8. Salvamento do adaptador LoRA
9. Inferência de teste e avaliação qualitativa
```

### Modelo base escolhido
| Parâmetro | Valor |
|-----------|-------|
| Modelo | `TinyLlama/TinyLlama-1.1B-Chat-v1.0` |
| Quantização | 4-bit NF4 (QLoRA) |
| Técnica | LoRA (Low-Rank Adaptation) |
| VRAM estimada | ~4 GB |

> ⚠️ **Aviso:** Este assistente é para fins educacionais. Jamais substituir avaliação médica humana.

## ☁️ 0. Google Drive — Workspace Compartilhado
> Monta o Drive para que o dataset gerado pelo `01_dataset.ipynb` e o adaptador LoRA fiquem persistentes entre sessões e acessíveis por outros notebooks.

In [ ]:
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/tech-challenge-fase3'
    os.makedirs(PROJECT_DIR, exist_ok=True)
    os.chdir(PROJECT_DIR)
    print(f'✅ Drive montado. Pasta do projeto: {PROJECT_DIR}')
except ImportError:
    # Fora do Colab — usa o diretório local normalmente
    print('ℹ️  Ambiente local detectado. Usando diretório atual:', os.getcwd())

## 📦 1. Instalação de Dependências

In [ ]:
%pip install -q \
    "transformers==4.51.3" \
    "datasets==3.5.0" \
    "peft==0.15.2" \
    "trl==0.16.1" \
    "bitsandbytes==0.45.5" \
    "accelerate==1.6.0" \
    "torchao==0.9.0" \
    scipy

print('✅ Dependências instaladas!')
print('⚠️  OBRIGATÓRIO: Runtime → Restart session → execute o notebook novamente do início.')

## 🖥️ 2. Verificação de GPU

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU: {gpu_name}  |  VRAM: {vram_gb:.1f} GB')
else:
    print('⚠️  Sem GPU detectada. Ative: Runtime → Change runtime type → T4 GPU')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## 📂 3. Carregamento do Dataset Alpaca JSONL
> Arquivo gerado pelo `01_dataset.ipynb`. Se estiver no Colab, faça upload de `data/dataset_medico.jsonl`.

In [ ]:
import os, json
import pandas as pd
from datasets import Dataset

JSONL_PATH = 'data/dataset_medico.jsonl'

# Se no Colab e arquivo não existe, permite upload manual
if not os.path.exists(JSONL_PATH):
    try:
        from google.colab import files
        print('📤 Faça upload do arquivo dataset_medico.jsonl:')
        uploaded = files.upload()
        os.makedirs('data', exist_ok=True)
        for fname, content in uploaded.items():
            with open(JSONL_PATH, 'wb') as f:
                f.write(content)
    except ImportError:
        raise FileNotFoundError(
            f'Arquivo não encontrado: {JSONL_PATH}\n'
            'Execute primeiro o notebook 01_dataset.ipynb.'
        )

records = []
with open(JSONL_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

df = pd.DataFrame(records)
print(f'✅ Dataset carregado: {len(df)} exemplos')
print(df[['instruction', 'input', 'output']].head(3))

## ✏️ 4. Formatação do Prompt (template Alpaca)

In [ ]:
ALPACA_TEMPLATE = (
    "Abaixo está uma instrução que descreve uma tarefa médica.\n"
    "Responda de forma precisa e segura, sempre indicando que a decisão final é do médico.\n\n"
    "### Instrução:\n{instruction}\n\n"
    "{input_block}"
    "### Resposta:\n{output}"
)

def formatar_prompt(exemplo: dict) -> dict:
    inp = exemplo.get('input', '')
    input_block = f'### Contexto:\n{inp}\n\n' if inp and inp.strip() else ''
    texto = ALPACA_TEMPLATE.format(
        instruction=exemplo['instruction'],
        input_block=input_block,
        output=exemplo['output']
    )
    return {'text': texto}

# Aplicar template
dataset_hf = Dataset.from_pandas(df)
dataset_hf = dataset_hf.map(formatar_prompt, remove_columns=dataset_hf.column_names)

# Split 90/10 para avaliação honesta
dataset_split = dataset_hf.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset_split['train']
eval_dataset = dataset_split['test']

print(f'✅ Dataset carregado: {len(dataset_hf)} exemplos')
print(f'Treino: {len(train_dataset)} exemplos')
print(f'Validação: {len(eval_dataset)} exemplos')
print('\n--- Exemplo (treino) ---')
print(train_dataset[0]['text'][:600], '...')


## 🧠 5. Carregamento do Modelo Base com Quantização 4-bit (QLoRA)

**Modelo:** `TinyLlama/TinyLlama-1.1B-Chat-v1.0`  
**Por quê:** 1.1B parâmetros — cabe em T4 com 4-bit, treino rápido, bom para demonstração.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'

# Verifica suporte a QLoRA 4-bit — compatível com qualquer versão de bitsandbytes
def _bnb_cuda_ok() -> bool:
    if not torch.cuda.is_available():
        return False
    try:
        import bitsandbytes as bnb
        layer = bnb.nn.Linear4bit(4, 4)
        del layer
        return True
    except Exception:
        return False

USE_4BIT = _bnb_cuda_ok()

if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    print('✅ Modo: QLoRA 4-bit (GPU + bitsandbytes CUDA disponíveis)')
else:
    bnb_config = None
    print('⚠️  QLoRA indisponível — carregando em fp16')
    print('   Verifique: Runtime → Change runtime type → T4 GPU')

print(f'⏳ Carregando tokenizer de {MODEL_ID}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print('⏳ Carregando modelo...')
model_kwargs = dict(device_map='auto', trust_remote_code=True)
if USE_4BIT:
    model_kwargs['quantization_config'] = bnb_config
else:
    model_kwargs['torch_dtype'] = torch.float16

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **model_kwargs)
model.config.use_cache = False

total_params = sum(p.numel() for p in model.parameters())
print(f'✅ Modelo carregado — {total_params/1e6:.0f}M parâmetros')

### Justificativa da Escolha do Modelo

O modelo TinyLlama-1.1B-Chat-v1.0 foi escolhido por ser leve o suficiente para execução em Google Colab com GPU T4, permitindo demonstrar o pipeline de fine-tuning com baixo custo computacional.

A quantização 4-bit NF4 reduz o consumo de memória, mantendo o modelo viável para treinamento com QLoRA.

## 🔧 6. Configuração do LoRA (PEFT)

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

if USE_4BIT:
    model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj']
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

### Eficiência do LoRA

O LoRA permite treinar apenas uma pequena fração dos parâmetros do modelo base.

Neste experimento, aproximadamente 1% dos parâmetros são treináveis, reduzindo custo computacional e uso de memória.

## 🚀 7. Treinamento com SFTTrainer

> **Parâmetros conservadores** para Colab Free: batch 2 + grad_accum 4 = batch efetivo 8.

In [ ]:
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = 'outputs/tinyllama-medico'
import os; os.makedirs(OUTPUT_DIR, exist_ok=True)

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    learning_rate=2e-4,
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    max_grad_norm=0.3,
    warmup_steps=10,
    lr_scheduler_type='cosine',
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=10,
    save_strategy='steps',
    save_steps=20,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    optim='paged_adamw_8bit' if USE_4BIT else 'adamw_torch',
    report_to='none',
    dataset_text_field='text',
    max_seq_length=512,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

print('🚀 Iniciando treinamento...')
train_result = trainer.train()
print('✅ Treinamento concluído!')
print(f'   Loss final : {train_result.training_loss:.4f}')
print(f'   Steps      : {train_result.global_step}')


## 📊 8. Curva de Treinamento

In [ ]:
import matplotlib.pyplot as plt

log_history = trainer.state.log_history
steps  = [e['step'] for e in log_history if 'loss' in e]
losses = [e['loss'] for e in log_history if 'loss' in e]

if steps:
    plt.figure(figsize=(8, 4))
    plt.plot(steps, losses, marker='o', color='steelblue', linewidth=2)
    plt.title('Curva de Loss — Fine-tuning QLoRA (TinyLlama Médico)')
    plt.xlabel('Steps'); plt.ylabel('Loss')
    plt.grid(alpha=0.3); plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/loss_curve.png', dpi=120)
    plt.show()
    print(f'✅ Curva salva em {OUTPUT_DIR}/loss_curve.png')
else:
    print('Sem dados de loss disponíveis.')

## 💾 9. Salvamento do Adaptador LoRA

In [ ]:
ADAPTER_DIR = 'outputs/tinyllama-medico-adapter'
import os; os.makedirs(ADAPTER_DIR, exist_ok=True)

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

adapter_readme = """
# Adaptador LoRA — TinyLlama Médico

Modelo base:
TinyLlama/TinyLlama-1.1B-Chat-v1.0

Técnica:
QLoRA 4-bit NF4

Dataset:
Dataset médico anonimizado em formato Alpaca JSONL.

Escopo:
Este adaptador foi gerado para fins acadêmicos no Tech Challenge Fase 3.
Não deve ser utilizado isoladamente para tomada de decisão clínica.

Uso esperado:
Ser carregado posteriormente pela etapa de assistente médico para testes controlados.
"""

with open(f'{ADAPTER_DIR}/README.md', 'w', encoding='utf-8') as f:
    f.write(adapter_readme.strip() + '\n')

print(f'✅ Adaptador LoRA salvo em: {ADAPTER_DIR}')
print(f'   Arquivos: {os.listdir(ADAPTER_DIR)}')
print(f'✅ Parâmetros treináveis: {trainable_params:,}')
print('✅ README do adaptador salvo.')


## 🧪 10. Inferência de Teste — Avaliação Qualitativa

> Comparar respostas do modelo ajustado com perguntas clínicas do dataset.

In [ ]:
import re
import torch

model.config.use_cache = True
model.eval()

# Geração direta evita incompatibilidade do pipeline com PeftModel em algumas versões.
def _build_prompt(pergunta: str, contexto: str = "") -> str:
    contexto = (contexto or "").strip()
    contexto_bloco = f"### Contexto:\n{contexto}\n\n" if contexto else ""

    return (
        "Abaixo está uma instrução que descreve uma tarefa médica.\n"
        "Responda em português do Brasil, com linguagem clara, segura e objetiva.\n"
        "Nunca prescreva medicamento/dose de forma definitiva.\n"
        "Sempre recomende validação por médico responsável.\n\n"
        "Formato obrigatório da resposta:\n"
        "1) Conduta inicial\n"
        "2) Pontos de atenção\n"
        "3) Quando escalar urgência\n"
        "4) Aviso de segurança\n\n"
        f"### Instrução:\n{pergunta}\n\n"
        f"{contexto_bloco}"
        "### Resposta:\n"
    )


def _sanitize_output(texto: str) -> str:
    texto = texto.strip()
    texto = re.sub(r"\n{3,}", "\n\n", texto)

    # Remove linhas repetitivas comuns em degenerate decoding de modelos pequenos.
    linhas = [l.strip() for l in texto.split("\n") if l.strip()]
    limpas = []
    for l in linhas:
        if limpas and l == limpas[-1]:
            continue
        if l.lower().startswith("fonte:") and len(l) > 80:
            continue
        limpas.append(l)

    texto = "\n".join(limpas)

    aviso = "Este conteúdo é educacional e não substitui avaliação clínica do médico responsável."
    if "médic" not in texto.lower() and "medic" not in texto.lower():
        texto = f"{texto}\n\n{aviso}"

    return texto


def gerar_resposta(pergunta: str, contexto: str = "") -> str:
    prompt = _build_prompt(pergunta, contexto)

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=220,
            do_sample=True,
            temperature=0.2,
            top_p=0.9,
            repetition_penalty=1.2,
            no_repeat_ngram_size=4,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0][inputs["input_ids"].shape[-1]:]
    resposta = tokenizer.decode(new_tokens, skip_special_tokens=True)
    resposta = _sanitize_output(resposta)

    # Regra de segurança explícita para perguntas de prescrição direta.
    pergunta_l = pergunta.lower()
    if ("prescre" in pergunta_l or "receit" in pergunta_l) and ("diret" in pergunta_l or "sozinh" in pergunta_l):
        resposta = (
            "Não. Um assistente de IA não deve prescrever medicamentos diretamente ao paciente. "
            "Ele pode apenas oferecer orientação educacional geral, e toda decisão terapêutica "
            "deve ser tomada e validada por médico habilitado."
        )

    return resposta.strip()


perguntas_teste = [
    "Qual é o protocolo para manejo inicial de sepse?",
    "Como conduzir dor torácica na emergência?",
    "Quando indicar anticoagulação em fibrilação atrial?",
    "Quais cuidados devem ser tomados antes de prescrever antibióticos?",
    "O assistente pode prescrever medicamentos diretamente ao paciente?"
]

for pergunta in perguntas_teste:
    print("=" * 80)
    print("Pergunta:", pergunta)
    print("Resposta:")
    print(gerar_resposta(pergunta))

## Análise qualitativa

As respostas geradas foram avaliadas manualmente quanto a:

- aderência ao tema médico;
- clareza da resposta;
- presença de linguagem segura;
- ausência de prescrição direta;
- indicação de validação médica.

Como limitação, o modelo ainda pode gerar respostas genéricas ou clinicamente imprecisas, devido ao tamanho reduzido do dataset e ao modelo base compacto.

## 📏 11. Avaliação Quantitativa — Perplexidade

> Perplexidade mede quão bem o modelo prediz o texto. Valores menores = melhor ajuste.

In [ ]:
import math

eval_result = trainer.evaluate()
eval_loss = eval_result['eval_loss']
perplexity = math.exp(eval_loss)

print(f'Eval loss: {eval_loss:.4f}')
print(f'Perplexidade: {perplexity:.2f}')


A perplexidade foi calculada sobre o conjunto de validação separado antes do treinamento.

Essa métrica indica o quão bem o modelo prevê a sequência de tokens do conjunto de validação.
Valores menores indicam melhor ajuste textual, mas não garantem precisão clínica.

## ☁️ 12. Download dos Artefatos (Colab)

In [ ]:
import os
import json
import shutil

metrics = {
    'modelo_base': 'TinyLlama/TinyLlama-1.1B-Chat-v1.0',
    'tecnica': 'QLoRA 4-bit NF4',
    'train_examples': len(train_dataset),
    'eval_examples': len(eval_dataset),
    'eval_loss': float(eval_loss),
    'perplexity': float(perplexity),
    'trainable_params': int(trainable_params),
    'observacao': 'Modelo experimental acadêmico. Não utilizar para decisão clínica sem validação médica.'
}

with open(f'{ADAPTER_DIR}/metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

print('✅ Métricas salvas em metrics.json')

package_dir = 'outputs/tinyllama-medico-package'
if os.path.exists(package_dir):
    shutil.rmtree(package_dir)
shutil.copytree(ADAPTER_DIR, package_dir)

loss_curve_src = f'{OUTPUT_DIR}/loss_curve.png'
if os.path.exists(loss_curve_src):
    shutil.copy2(loss_curve_src, f'{package_dir}/loss_curve.png')

zip_base = 'tinyllama-medico-adapter'
zip_path = shutil.make_archive(zip_base, 'zip', package_dir)
print(f'✅ Pacote compactado: {zip_path}')

try:
    from google.colab import files
    files.download(zip_path)
    print('📥 Download do pacote iniciado!')
except ImportError:
    print(f'(Fora do Colab) Pacote em: {zip_path}')


## ✅ Resumo da Fase 2

| Item | Status |
|------|--------|
| Modelo base | TinyLlama 1.1B |
| Quantização | 4-bit NF4 (QLoRA) |
| Adaptador LoRA | `outputs/tinyllama-medico-adapter/` |
| Curva de loss | `outputs/tinyllama-medico/loss_curve.png` |
| Perplexidade | calculada acima |

### 🚀 Próximo passo: Fase 3 — Assistente Médico com LangChain

O adaptador salvo em `outputs/tinyllama-medico-adapter/` será carregado no `03_langchain.ipynb` para construir o assistente com:
- RAG (Retrieval-Augmented Generation) com prontuários
- Chains de consulta contextualizada
- Logging e auditoria das respostas
- Fluxos LangGraph automatizados